<a href="https://colab.research.google.com/github/kamxsato/MIS444_Final_Project/blob/main/24104087_Final_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Libraries

In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import missingno as msno
import warnings
import scipy.stats as stats
from scipy.stats import (shapiro, levene, mannwhitneyu, kruskal,
                          ks_2samp, chi2_contingency, pearsonr, spearmanr)
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report, roc_curve)
from sklearn.feature_selection import RFECV

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', '{:.3f}'.format)
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)

print('✓ All libraries loaded')
print(f'pandas {pd.__version__} | numpy {np.__version__}')

✓ All libraries loaded
pandas 2.2.3 | numpy 2.1.3


# Data

In [15]:
df = pd.read_csv('OHI_OHI_WIDEF.csv', low_memory=False)
print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Memory usage: {df.memory_usage(deep=True).sum() / 1e6:.2f} MB')


Shape: 194 rows × 29 columns
Memory usage: 0.19 MB


In [16]:
print('\n=== FIRST 5 ROWS ===')
display(df.head())
print('\n=== LAST 5 ROWS ===')
display(df.tail())


=== FIRST 5 ROWS ===


,FREQ,FREQ_LABEL,REF_AREA,REF_AREA_LABEL,INDICATOR,INDICATOR_LABEL,UNIT_MEASURE,UNIT_MEASURE_LABEL,DATABASE_ID,DATABASE_ID_LABEL,UNIT_MULT,UNIT_MULT_LABEL,OBS_STATUS,OBS_STATUS_LABEL,OBS_CONF,OBS_CONF_LABEL,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
0,A,Annual,ABW,Aruba,OHI_OHI_GIS,Ocean Health Global Index Scores,IX_0T100,Score 0-100,OHI_OHI,Ocean Health Index (OHI),0,Units,A,Normal value,PU,Public,70.790,70.100,70.090,70.520,71.280,72.140,77.650,80.870,79.940,79.390,76.430,64.900,67.500
1,A,Annual,AGO,Angola,OHI_OHI_GIS,Ocean Health Global Index Scores,IX_0T100,Score 0-100,OHI_OHI,Ocean Health Index (OHI),0,Units,A,Normal value,PU,Public,63.330,64.570,64.570,67.000,67.660,66.700,65.710,61.900,59.760,59.230,59.410,57.910,57.890
2,A,Annual,AIA,Anguilla,OHI_OHI_GIS,Ocean Health Global Index Scores,IX_0T100,Score 0-100,OHI_OHI,Ocean Health Index (OHI),0,Units,A,Normal value,PU,Public,72.170,70.600,70.820,70.860,70.740,70.210,70.360,70.300,70.360,70.350,70.440,70.630,70.870
3,A,Annual,ALB,Albania,OHI_OHI_GIS,Ocean Health Global Index Scores,IX_0T100,Score 0-100,OHI_OHI,Ocean Health Index (OHI),0,Units,A,Normal value,PU,Public,62.950,63.270,63.860,66.550,66.160,66.700,67.380,67.520,71.790,71.700,71.190,65.380,69.680
4,A,Annual,ANT,Netherlands Antilles,OHI_OHI_GIS,Ocean Health Global Index Scores,IX_0T100,Score 0-100,OHI_OHI,Ocean Health Index (OHI),0,Units,A,Normal value,PU,Public,74.660,74.980,74.970,74.920,77.450,77.450,77.440,77.440,76.610,76.510,76.330,76.350,76.340



=== LAST 5 ROWS ===


,FREQ,FREQ_LABEL,REF_AREA,REF_AREA_LABEL,INDICATOR,INDICATOR_LABEL,UNIT_MEASURE,UNIT_MEASURE_LABEL,DATABASE_ID,DATABASE_ID_LABEL,UNIT_MULT,UNIT_MULT_LABEL,OBS_STATUS,OBS_STATUS_LABEL,OBS_CONF,OBS_CONF_LABEL,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
189,A,Annual,WLD,World,OHI_OHI_GIS,Ocean Health Global Index Scores,IX_0T100,Score 0-100,OHI_OHI,Ocean Health Index (OHI),0,Units,A,Normal value,PU,Public,71.460,72.240,72.880,73.360,73.480,73.540,73.660,73.800,73.870,73.690,73.920,69.030,68.780
190,A,Annual,WLF,Wallis-et-Futuna (Fr.),OHI_OHI_GIS,Ocean Health Global Index Scores,IX_0T100,Score 0-100,OHI_OHI,Ocean Health Index (OHI),0,Units,A,Normal value,PU,Public,73.670,74.650,74.600,74.470,74.300,74.120,73.950,73.810,78.040,77.560,77.440,77.440,77.270
191,A,Annual,WSM,Samoa,OHI_OHI_GIS,Ocean Health Global Index Scores,IX_0T100,Score 0-100,OHI_OHI,Ocean Health Index (OHI),0,Units,A,Normal value,PU,Public,71.950,72.240,72.370,72.600,71.570,71.650,72.040,72.460,74.990,74.990,74.990,66.150,64.710
192,A,Annual,YEM,"Yemen, Rep.",OHI_OHI_GIS,Ocean Health Global Index Scores,IX_0T100,Score 0-100,OHI_OHI,Ocean Health Index (OHI),0,Units,A,Normal value,PU,Public,70.690,73.130,72.600,72.850,73.580,73.200,67.120,66.640,66.010,66.110,67.010,65.000,64.790
193,A,Annual,ZAF,South Africa,OHI_OHI_GIS,Ocean Health Global Index Scores,IX_0T100,Score 0-100,OHI_OHI,Ocean Health Index (OHI),0,Units,A,Normal value,PU,Public,70.270,72.190,71.520,70.740,70.130,68.960,67.580,68.750,69.010,69.160,69.190,61.740,61.370


In [17]:
df.info(verbose=True, show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 194 entries, 0 to 193
Data columns (total 29 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   FREQ                194 non-null    object 
 1   FREQ_LABEL          194 non-null    object 
 2   REF_AREA            194 non-null    object 
 3   REF_AREA_LABEL      194 non-null    object 
 4   INDICATOR           194 non-null    object 
 5   INDICATOR_LABEL     194 non-null    object 
 6   UNIT_MEASURE        194 non-null    object 
 7   UNIT_MEASURE_LABEL  194 non-null    object 
 8   DATABASE_ID         194 non-null    object 
 9   DATABASE_ID_LABEL   194 non-null    object 
 10  UNIT_MULT           194 non-null    int64  
 11  UNIT_MULT_LABEL     194 non-null    object 
 12  OBS_STATUS          194 non-null    object 
 13  OBS_STATUS_LABEL    194 non-null    object 
 14  OBS_CONF            194 non-null    object 
 15  OBS_CONF_LABEL      194 non-null    object 
 16  2012    

In [18]:
overview = pd.DataFrame({
    'dtype': df.dtypes,
    'nunique': df.nunique(),
    'nulls': df.isnull().sum(),
    'null_%': (df.isnull().mean()*100).round(2),
    'sample': [df[c].dropna().iloc[0] if df[c].notna().any() else 'ALL NULL' for c in df.columns]
})
display(overview)

,dtype,nunique,nulls,null_%,sample
FREQ,object,1,0,0.000,A
FREQ_LABEL,object,1,0,0.000,Annual
REF_AREA,object,194,0,0.000,ABW
REF_AREA_LABEL,object,194,0,0.000,Aruba
INDICATOR,object,1,0,0.000,OHI_OHI_GIS
INDICATOR_LABEL,object,1,0,0.000,Ocean Health Global Index Scores
UNIT_MEASURE,object,1,0,0.000,IX_0T100
UNIT_MEASURE_LABEL,object,1,0,0.000,Score 0-100
DATABASE_ID,object,1,0,0.000,OHI_OHI
DATABASE_ID_LABEL,object,1,0,0.000,Ocean Health Index (OHI)


In [19]:
n_dupes = df.duplicated().sum()
print(f'Full duplicate rows: {n_dupes} ({n_dupes/len(df)*100:.2f}%)')

Full duplicate rows: 0 (0.00%)


# Preprocessing

In [20]:
#Remove Metadata columns & fix year column
metadata_cols = ['FREQ', 'FREQ_LABEL', 'REF_AREA', 'REF_AREA_LABEL', 'INDICATOR', 'INDICATOR_LABEL', 'UNIT_MEASURE',
                 'UNIT_MEASURE_LABEL', 'DATABASE_ID', 'DATABASE_ID_LABEL', 'UNIT_MULT', 'UNIT_MULT_LABEL', 'OBS_STATUS',
                 'OBS_STATUS_LABEL', 'OBS_CONF', 'OBS_CONF_LABEL']

year_cols = ['2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024']

In [22]:
df_clean = df[['REF_AREA', 'REF_AREA_LABEL'] + year_cols].copy()
df_clean.rename(columns={'REF_AREA': 'country_code', 'REF_AREA_LABEL': 'country'}, inplace=True)
print(f'Cleaned shape: {df_clean.shape}')
df_clean

Cleaned shape: (194, 15)


,country_code,country,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
0,ABW,Aruba,70.790,70.100,70.090,70.520,71.280,72.140,77.650,80.870,79.940,79.390,76.430,64.900,67.500
1,AGO,Angola,63.330,64.570,64.570,67.000,67.660,66.700,65.710,61.900,59.760,59.230,59.410,57.910,57.890
2,AIA,Anguilla,72.170,70.600,70.820,70.860,70.740,70.210,70.360,70.300,70.360,70.350,70.440,70.630,70.870
3,ALB,Albania,62.950,63.270,63.860,66.550,66.160,66.700,67.380,67.520,71.790,71.700,71.190,65.380,69.680
4,ANT,Netherlands Antilles,74.660,74.980,74.970,74.920,77.450,77.450,77.440,77.440,76.610,76.510,76.330,76.350,76.340
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
189,WLD,World,71.460,72.240,72.880,73.360,73.480,73.540,73.660,73.800,73.870,73.690,73.920,69.030,68.780
190,WLF,Wallis-et-Futuna (Fr.),73.670,74.650,74.600,74.470,74.300,74.120,73.950,73.810,78.040,77.560,77.440,77.440,77.270
191,WSM,Samoa,71.950,72.240,72.370,72.600,71.570,71.650,72.040,72.460,74.990,74.990,74.990,66.150,64.710
192,YEM,"Yemen, Rep.",70.690,73.130,72.600,72.850,73.580,73.200,67.120,66.640,66.010,66.110,67.010,65.000,64.790


In [27]:
#Missing values
df_clean.isna().sum()

,0
country_code,0
country,0
2012,0
2013,0
2014,0
2015,0
2016,0
2017,0
2018,0
2019,0


In [29]:
#Create target
def categorize_risk(score):
    if pd.isna(score):
        return 'Unknown'
    elif score < 60:
        return 'Critical'
    elif score < 75:
        return 'Moderate'
    else:
        return 'Good'

df_clean['risk_category'] = df_clean['2024'].apply(categorize_risk)
df_clean.head()

,country_code,country,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,risk_category
0,ABW,Aruba,70.790,70.100,70.090,70.520,71.280,72.140,77.650,80.870,79.940,79.390,76.430,64.900,67.500,Moderate
1,AGO,Angola,63.330,64.570,64.570,67.000,67.660,66.700,65.710,61.900,59.760,59.230,59.410,57.910,57.890,Critical
2,AIA,Anguilla,72.170,70.600,70.820,70.860,70.740,70.210,70.360,70.300,70.360,70.350,70.440,70.630,70.870,Moderate
3,ALB,Albania,62.950,63.270,63.860,66.550,66.160,66.700,67.380,67.520,71.790,71.700,71.190,65.380,69.680,Moderate
4,ANT,Netherlands Antilles,74.660,74.980,74.970,74.920,77.450,77.450,77.440,77.440,76.610,76.510,76.330,76.350,76.340,Good


In [30]:
#Remove category: Unknown
df_clean = df_clean[df_clean['risk_category'] != 'Unknown']
print(f'Risk category distribution:')
print(df_clean['risk_category'].value_counts())

Risk category distribution:
risk_category
Moderate    144
Critical     28
Good         22
Name: count, dtype: int64
